In [54]:
import numpy as np
import pandas as pd
from scipy.stats import skew, kurtosis
from tqdm import tqdm
from pathlib import Path
from data_loader import build_complete_dataset
from window import create_record_windows
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix)


In [55]:
def extract_window_features(window_df, fs, prefix=""):
    features = {}
    columns = [column for column in window_df.columns if column != "Time" ]
    data = window_df.drop(columns=["Time"], errors="ignore").values
    for i, col in enumerate(columns):
            signal = data[:, i]
            features[f"{prefix}_{col}_mean"] = np.mean(signal)
            features[f"{prefix}_{col}_std"] = np.std(signal)
            features[f"{prefix}_{col}_rms"] = np.sqrt(np.mean(signal**2))
            features[f"{prefix}_{col}_max"] = np.max(signal)
            features[f"{prefix}_{col}_min"] = np.min(signal)
            features[f"{prefix}_{col}_ptp"] = np.ptp(signal)
            features[f"{prefix}_{col}_skew"] = skew(signal) if len(signal) > 3 else 0
            features[f"{prefix}_{col}_kurtosis"] = kurtosis(signal) if len(signal) > 3 else 0
            fft_vals = np.abs(np.fft.rfft(signal))
            freqs = np.fft.rfftfreq(len(signal), d=1 / fs)
            features[f"{prefix}_{col}_fft_energy"] = np.sum(fft_vals**2)
            features[f"{prefix}_{col}_fft_peak_freq"] = freqs[np.argmax(fft_vals)]
    return features

In [56]:
def process_record_features(record, create_record_windows_func):
    acc_windows, gyro_windows, mic_windows = create_record_windows_func(record)
    
    record_features = []
    min_len = min(len(acc_windows), len(gyro_windows), len(mic_windows))
    
    for i in range(min_len):
        features = {}
        features.update(extract_window_features(acc_windows[i], fs=6700, prefix="acc"))
        features.update(extract_window_features(gyro_windows[i], fs=6700, prefix="gyro"))
        features.update(extract_window_features(mic_windows[i], fs=16000, prefix="mic"))
        
        features["segment_id"] = record.metadata.get("segment_id")
        features["split_label"] = record.metadata.get("split_label")
        features["anomaly_label"] = record.metadata.get("anomaly_label")
        features["domain_shift_op"] = record.metadata.get("domain_shift_op")
        features["domain_shift_env"] = record.metadata.get("domain_shift_env")
        record_features.append(features)
        
    return pd.DataFrame(record_features)

In [57]:
def build_feature_dataset(dataset, create_record_windows_func):
    results = []
    for record in tqdm(dataset, desc="Ekstrakcja cech"):
        df_feats = process_record_features(record, create_record_windows_func)
        results.append(df_feats)
    return pd.concat(results, ignore_index=True)

In [58]:
path = Path("../data")
train_path = path / "X_train.csv"
test_path = path / "X_test.csv"

if train_path.exists() and test_path.exists():
    X_train_df = pd.read_csv(train_path)
    X_test_df = pd.read_csv(test_path)
else:
    df = build_complete_dataset()
    X_train_df = build_feature_dataset(df[0], create_record_windows)
    X_test_df = build_feature_dataset(df[1], create_record_windows)
    X_train_df.to_csv(train_path, index=False)
    X_test_df.to_csv(test_path, index=False)

print("Train:", X_train_df.shape)
print("Test :", X_test_df.shape)

print(X_train_df.columns.tolist())
print(X_test_df.columns.tolist())


Train: (102984, 75)
Test : (25984, 75)
['acc_A_x [g]_mean', 'acc_A_x [g]_std', 'acc_A_x [g]_rms', 'acc_A_x [g]_max', 'acc_A_x [g]_min', 'acc_A_x [g]_ptp', 'acc_A_x [g]_skew', 'acc_A_x [g]_kurtosis', 'acc_A_x [g]_fft_energy', 'acc_A_x [g]_fft_peak_freq', 'acc_A_y [g]_mean', 'acc_A_y [g]_std', 'acc_A_y [g]_rms', 'acc_A_y [g]_max', 'acc_A_y [g]_min', 'acc_A_y [g]_ptp', 'acc_A_y [g]_skew', 'acc_A_y [g]_kurtosis', 'acc_A_y [g]_fft_energy', 'acc_A_y [g]_fft_peak_freq', 'acc_A_z [g]_mean', 'acc_A_z [g]_std', 'acc_A_z [g]_rms', 'acc_A_z [g]_max', 'acc_A_z [g]_min', 'acc_A_z [g]_ptp', 'acc_A_z [g]_skew', 'acc_A_z [g]_kurtosis', 'acc_A_z [g]_fft_energy', 'acc_A_z [g]_fft_peak_freq', 'gyro_G_x [mdps]_mean', 'gyro_G_x [mdps]_std', 'gyro_G_x [mdps]_rms', 'gyro_G_x [mdps]_max', 'gyro_G_x [mdps]_min', 'gyro_G_x [mdps]_ptp', 'gyro_G_x [mdps]_skew', 'gyro_G_x [mdps]_kurtosis', 'gyro_G_x [mdps]_fft_energy', 'gyro_G_x [mdps]_fft_peak_freq', 'gyro_G_y [mdps]_mean', 'gyro_G_y [mdps]_std', 'gyro_G_y [mdps]_

In [59]:
X_train_df_source = X_train_df[X_train_df["split_label"] == "Normal_Source_Train"].copy()
X_test_df_source = X_test_df[X_test_df["split_label"].isin(["Normal_Source_Test", "Anomaly_Source_Test"])].copy()

X_train_df_target = X_train_df[X_train_df["split_label"] == "Normal_Target_Train"].copy()
X_test_df_target = X_test_df[X_test_df["split_label"].isin(["Normal_Target_Test", "Anomaly_Target_Test"])].copy()

In [60]:
metadata_cols = ["segment_id", "anomaly_label", "domain_shift_op", "domain_shift_env"]

acc_features = [col for col in X_train_df.columns if col.startswith("acc_")]
gyro_features = [col for col in X_train_df.columns if col.startswith("gyro_")]
mic_features = [col for col in X_train_df.columns if col.startswith("mic_")]

In [61]:
X_train_acc = X_train_df[acc_features].fillna(0)
X_test_acc = X_test_df[acc_features].fillna(0)

X_train_mic = X_train_df[mic_features].fillna(0)
X_test_mic = X_test_df[mic_features].fillna(0)

X_train_gyro = X_train_df[gyro_features].fillna(0)
X_test_gyro = X_test_df[gyro_features].fillna(0)

X_train_source_acc = X_train_df_source[acc_features].fillna(0)
X_train_source_mic = X_train_df_source[mic_features].fillna(0)
X_train_source_gyro = X_train_df_source[gyro_features].fillna(0)

X_test_source_acc = X_test_df_source[acc_features].fillna(0)
X_test_source_mic = X_test_df_source[mic_features].fillna(0)
X_test_source_gyro = X_test_df_source[gyro_features].fillna(0)

X_train_target_acc = X_train_df_target[acc_features].fillna(0)
X_train_target_mic = X_train_df_target[mic_features].fillna(0)
X_train_target_gyro = X_train_df_target[gyro_features].fillna(0)

X_test_target_acc = X_test_df_target[acc_features].fillna(0)
X_test_target_mic = X_test_df_target[mic_features].fillna(0)
X_test_target_gyro = X_test_df_target[gyro_features].fillna(0)


In [62]:
def run_isolation_forest(X_train, X_test, train_df, test_df, sensor_type):
    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    model = IsolationForest(
        n_estimators=300,
        contamination='auto',
        max_features=1.0,
        max_samples=1024,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train_scaled)

    test_scores = -model.score_samples(X_test_scaled)
    train_scores = -model.score_samples(X_train_scaled)

    results = pd.DataFrame({"segment_id": test_df["segment_id"].values,
        "anomaly_label": test_df["anomaly_label"].values,
        "score": test_scores})

    segment_scores = (results.groupby("segment_id").agg(score=("score", "max"), anomaly_label=("anomaly_label", "first")).reset_index())
    segment_scores["y_true"] = (segment_scores["anomaly_label"] == "loosescrewsA").astype(int)

    finite = (pd.Series(train_scores).replace([np.inf, -np.inf], np.nan).dropna().values)

    threshold = (np.mean(finite) + 2 * np.std(finite) if finite.size else np.inf)

    segment_scores["y_pred"] = (segment_scores["score"] >= threshold).astype(int)

    y_true = segment_scores["y_true"]
    y_pred = segment_scores["y_pred"]

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    roc_auc = roc_auc_score(y_true, segment_scores["score"]) 

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    print(f"{sensor_type}")
    print(f"Threshold: {threshold:.4f}")
    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1-score : {f1:.4f}")
    print(f"ROC AUC  : {roc_auc:.4f}")
    print(cm)

    metrics = {"Sensors": sensor_type,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "ROC_AUC": roc_auc,
        "TN": cm[0, 0],
        "FP": cm[0, 1],
        "FN": cm[1, 0],
        "TP": cm[1, 1]}

    return metrics, segment_scores, model, scaler, threshold

trenowanie i testowanie osobnych modeli wyłącznie na domenie source

In [63]:
metrics_acc_source, pred_acc_source, model_acc_source, scaler_acc_source, threshold_acc_source = run_isolation_forest(X_train_source_acc,
        X_test_source_acc,
        X_train_df_source,
        X_test_df_source,
        "ACC - source")

metrics_mic_source, pred_mic_source, model_mic_source, scaler_mic_source, threshold_mic_source = run_isolation_forest(
        X_train_source_mic,
        X_test_source_mic,
        X_train_df_source,
        X_test_df_source,
        "MIC - source")

metrics_gyro_source, pred_gyro_source, model_gyro_source, scaler_gyro_source, threshold_gyro_source = run_isolation_forest(
        X_train_source_gyro,
        X_test_source_gyro,
        X_train_df_source,
        X_test_df_source,
        "GYRO - source")

ACC - source
Threshold: 0.5122
Accuracy : 0.6336
Precision: 0.6013
Recall   : 0.7931
F1-score : 0.6840
ROC AUC  : 0.6659
[[55 61]
 [24 92]]
MIC - source
Threshold: 0.4969
Accuracy : 0.5905
Precision: 0.5652
Recall   : 0.7845
F1-score : 0.6570
ROC AUC  : 0.5710
[[46 70]
 [25 91]]
GYRO - source
Threshold: 0.4882
Accuracy : 0.5560
Precision: 0.5631
Recall   : 0.5000
F1-score : 0.5297
ROC AUC  : 0.5632
[[71 45]
 [58 58]]


segment jest klasyfikowany jako anomalia, jeśli co najmniej jeden z trzech modeli wykryje anomalię

In [64]:
final_source_results = (pred_acc_source[["segment_id", "y_true", "y_pred"]].rename(columns={"y_pred": "pred_acc"})
    .merge(pred_mic_source[["segment_id", "y_pred"]].rename(columns={"y_pred": "pred_mic"}), on="segment_id")
    .merge(pred_gyro_source[["segment_id", "y_pred"]].rename(columns={"y_pred": "pred_gyro"}), on="segment_id"))
final_source_results["y_pred_final"] = (final_source_results[["pred_acc", "pred_mic", "pred_gyro"]].sum(axis=1) >= 1).astype(int)
y_true_source = final_source_results["y_true"]
y_pred_source_final = final_source_results["y_pred_final"]

accuracy_source = accuracy_score(y_true_source, y_pred_source_final)
precision_source = precision_score(y_true_source, y_pred_source_final, zero_division=0)
recall_source = recall_score(y_true_source, y_pred_source_final, zero_division=0)
f1_source = f1_score(y_true_source, y_pred_source_final, zero_division=0)
cm_source = confusion_matrix(y_true_source, y_pred_source_final, labels=[0, 1])
print("Final Source Results (at least 1 sensor):")
print(f"Accuracy : {accuracy_source:.4f}")
print(f"Precision: {precision_source:.4f}")
print(f"Recall   : {recall_source:.4f}")
print(f"F1-score : {f1_source:.4f}")
print("Confusion matrix:")
print(cm_source)

Final Source Results (at least 1 sensor):
Accuracy : 0.5388
Precision: 0.5213
Recall   : 0.9483
F1-score : 0.6728
Confusion matrix:
[[ 15 101]
 [  6 110]]


segment jest klasyfikowany jako anomalia, jeśli co najmniej dwa z trzech modeli wykryją anomalię

In [65]:
final_source_results = (pred_acc_source[["segment_id", "y_true", "y_pred"]].rename(columns={"y_pred": "pred_acc"})
    .merge(pred_mic_source[["segment_id", "y_pred"]].rename(columns={"y_pred": "pred_mic"}), on="segment_id")
    .merge(pred_gyro_source[["segment_id", "y_pred"]].rename(columns={"y_pred": "pred_gyro"}), on="segment_id"))
final_source_results["y_pred_final"] = (final_source_results[["pred_acc", "pred_mic", "pred_gyro"]].sum(axis=1) >= 2).astype(int)
y_true_source = final_source_results["y_true"]
y_pred_source_final = final_source_results["y_pred_final"]

accuracy_source = accuracy_score(y_true_source, y_pred_source_final)
precision_source = precision_score(y_true_source, y_pred_source_final, zero_division=0)
recall_source = recall_score(y_true_source, y_pred_source_final, zero_division=0)
f1_source = f1_score(y_true_source, y_pred_source_final, zero_division=0)
cm_source = confusion_matrix(y_true_source, y_pred_source_final, labels=[0, 1])

print("Final Source Results (majority voting):")
print(f"Accuracy : {accuracy_source:.4f}")
print(f"Precision: {precision_source:.4f}")
print(f"Recall   : {recall_source:.4f}")
print(f"F1-score : {f1_source:.4f}")
print("Confusion matrix:")
print(cm_source)

Final Source Results (majority voting):
Accuracy : 0.6121
Precision: 0.5915
Recall   : 0.7241
F1-score : 0.6512
Confusion matrix:
[[58 58]
 [32 84]]


trenowanie osobnych modeli dla każdego czujnika wyłącznie z wykorzystaniem domeny source, a testowanie na target

In [66]:
metrics_acc, pred_acc, model_acc, scaler_acc, threshold_acc = run_isolation_forest(X_train_source_acc, X_test_target_acc,  X_train_df_source, X_test_df_target, "ACC source-> target")
metrics_mic, pred_mic, model_mic, scaler_mic, threshold_mic = run_isolation_forest(X_train_source_mic, X_test_target_mic, X_train_df_source, X_test_df_target, "MIC source-> target")
metrics_gyro, pred_gyro, model_gyro, scaler_gyro, threshold_gyro = run_isolation_forest( X_train_source_gyro, X_test_target_gyro, X_train_df_source, X_test_df_target, "GYRO source-> target")

ACC source-> target
Threshold: 0.5122
Accuracy : 0.5647
Precision: 0.5366
Recall   : 0.9483
F1-score : 0.6854
ROC AUC  : 0.4186
[[ 21  95]
 [  6 110]]
MIC source-> target
Threshold: 0.4969
Accuracy : 0.5517
Precision: 0.5600
Recall   : 0.4828
F1-score : 0.5185
ROC AUC  : 0.5742
[[72 44]
 [60 56]]
GYRO source-> target
Threshold: 0.4882
Accuracy : 0.4569
Precision: 0.4679
Recall   : 0.6293
F1-score : 0.5368
ROC AUC  : 0.4076
[[33 83]
 [43 73]]


segment jest klasyfikowany jako anomalia, jeśli co najmniej jeden z trzech modeli wykryje anomalię

In [67]:
final_results = (pred_acc[["segment_id", "y_true", "y_pred"]].rename(columns={"y_pred": "pred_acc"})
				 .merge(pred_mic[["segment_id", "y_pred"]].rename(columns={"y_pred": "pred_mic"}), on="segment_id")
				 .merge(pred_gyro[["segment_id", "y_pred"]].rename(columns={"y_pred": "pred_gyro"}), on="segment_id"))
final_results["y_pred_final"] = (final_results[["pred_acc", "pred_mic", "pred_gyro"]].sum(axis=1) >= 1).astype(int)

y_true = final_results["y_true"]
y_pred_final = final_results["y_pred_final"]

accuracy = accuracy_score(y_true, y_pred_final)
precision = precision_score(y_true, y_pred_final, zero_division=0)
recall = recall_score(y_true, y_pred_final, zero_division=0)
f1 = f1_score(y_true, y_pred_final, zero_division=0)
cm = confusion_matrix(y_true, y_pred_final, labels=[0, 1])

print("Final Results source -> target (at least 1 sensor):")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")
print("Confusion matrix:")
print(cm)

Final Results source -> target (at least 1 sensor):
Accuracy : 0.5216
Precision: 0.5110
Recall   : 1.0000
F1-score : 0.6764
Confusion matrix:
[[  5 111]
 [  0 116]]


teraz segment jest klasyfikowany jako anomalia, jeśli co najmniej dwa z trzech modeli wykryją anomalię

In [68]:
final_results = (pred_acc[["segment_id", "y_true", "y_pred"]].rename(columns={"y_pred": "pred_acc"})
				 .merge(pred_mic[["segment_id", "y_pred"]].rename(columns={"y_pred": "pred_mic"}), on="segment_id")
				 .merge(pred_gyro[["segment_id", "y_pred"]].rename(columns={"y_pred": "pred_gyro"}), on="segment_id"))
final_results["y_pred_final"] = (final_results[["pred_acc", "pred_mic", "pred_gyro"]].sum(axis=1) >= 2).astype(int)
y_true = final_results["y_true"]
y_pred_final = final_results["y_pred_final"]

accuracy = accuracy_score(y_true, y_pred_final)
precision = precision_score(y_true, y_pred_final, zero_division=0)
recall = recall_score(y_true, y_pred_final, zero_division=0)
f1 = f1_score(y_true, y_pred_final, zero_division=0)
cm = confusion_matrix(y_true, y_pred_final, labels=[0, 1])

print("Final Results source -> target (majority voting):")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")
print("Confusion matrix:")
print(cm)

Final Results source -> target (majority voting):
Accuracy : 0.5129
Precision: 0.5087
Recall   : 0.7586
F1-score : 0.6090
Confusion matrix:
[[31 85]
 [28 88]]


trenowanie i testowanie osobnych modeli dla każdego czujnika z wykorzystaniem całych zbiorów treningowego i testowego

In [69]:
metrics_acc, pred_acc, model_acc, scaler_acc, threshold_acc = run_isolation_forest(X_train_acc, X_test_acc,  X_train_df, X_test_df, "ACC")
metrics_mic, pred_mic, model_mic, scaler_mic, threshold_mic = run_isolation_forest(X_train_mic, X_test_mic, X_train_df, X_test_df, "MIC")
metrics_gyro, pred_gyro, model_gyro, scaler_gyro, threshold_gyro = run_isolation_forest( X_train_gyro, X_test_gyro, X_train_df, X_test_df, "GYRO")

ACC
Threshold: 0.5129
Accuracy : 0.5970
Precision: 0.5610
Recall   : 0.8922
F1-score : 0.6889
ROC AUC  : 0.5499
[[ 70 162]
 [ 25 207]]
MIC
Threshold: 0.4964
Accuracy : 0.5647
Precision: 0.5586
Recall   : 0.6164
F1-score : 0.5861
ROC AUC  : 0.5684
[[119 113]
 [ 89 143]]
GYRO
Threshold: 0.4858
Accuracy : 0.5065
Precision: 0.5062
Recall   : 0.5302
F1-score : 0.5179
ROC AUC  : 0.4977
[[112 120]
 [109 123]]


segment jest klasyfikowany jako anomalia, jeśli co najmniej jeden z trzech modeli wykryje anomalię

In [70]:
final_results = (pred_acc[["segment_id", "y_true", "y_pred"]].rename(columns={"y_pred": "pred_acc"})
				 .merge(pred_mic[["segment_id", "y_pred"]].rename(columns={"y_pred": "pred_mic"}), on="segment_id")
				 .merge(pred_gyro[["segment_id", "y_pred"]].rename(columns={"y_pred": "pred_gyro"}), on="segment_id"))
final_results["y_pred_final"] = (final_results[["pred_acc", "pred_mic", "pred_gyro"]].sum(axis=1) >= 1).astype(int)

y_true = final_results["y_true"]
y_pred_final = final_results["y_pred_final"]

accuracy = accuracy_score(y_true, y_pred_final)
precision = precision_score(y_true, y_pred_final, zero_division=0)
recall = recall_score(y_true, y_pred_final, zero_division=0)
f1 = f1_score(y_true, y_pred_final, zero_division=0)
cm = confusion_matrix(y_true, y_pred_final, labels=[0, 1])

print("Final Results (at least 1 sensor):")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")
print("Confusion matrix:")
print(cm)

Final Results (at least 1 sensor):
Accuracy : 0.5216
Precision: 0.5115
Recall   : 0.9612
F1-score : 0.6677
Confusion matrix:
[[ 19 213]
 [  9 223]]


teraz segment jest klasyfikowany jako anomalia, jeśli co najmniej dwa z trzech modeli wykryją anomalię

In [71]:
final_results = (pred_acc[["segment_id", "y_true", "y_pred"]].rename(columns={"y_pred": "pred_acc"})
				 .merge(pred_mic[["segment_id", "y_pred"]].rename(columns={"y_pred": "pred_mic"}), on="segment_id")
				 .merge(pred_gyro[["segment_id", "y_pred"]].rename(columns={"y_pred": "pred_gyro"}), on="segment_id"))
final_results["y_pred_final"] = (final_results[["pred_acc", "pred_mic", "pred_gyro"]].sum(axis=1) >= 2).astype(int)
y_true = final_results["y_true"]
y_pred_final = final_results["y_pred_final"]

accuracy = accuracy_score(y_true, y_pred_final)
precision = precision_score(y_true, y_pred_final, zero_division=0)
recall = recall_score(y_true, y_pred_final, zero_division=0)
f1 = f1_score(y_true, y_pred_final, zero_division=0)
cm = confusion_matrix(y_true, y_pred_final, labels=[0, 1])

print("Final Results (majority voting):")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")
print("Confusion matrix:")
print(cm)

Final Results (majority voting):
Accuracy : 0.5841
Precision: 0.5615
Recall   : 0.7672
F1-score : 0.6485
Confusion matrix:
[[ 93 139]
 [ 54 178]]
